# Test Notebook: $MFT Parsing and UsnJrnl Enrichment

## Objective
Parse $MFT to extract MACE timestamps and join with UsnJrnl data to enable timestamp discrepancy detection (Oh et al.'s core heuristic).

## Approach
1. Parse $MFT using analyzeMFT to extract file metadata
2. Load existing UsnJrnl.csv from Oh et al.'s tool
3. Join on FileReferenceNumber to add MACE timestamps
4. Verify timestamp discrepancy detection works

--

In [30]:
## Cell 1: Setup and Imports
import pandas as pd
import subprocess
import os
from pathlib import Path
from datetime import datetime

# Base paths
BASE_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis')
TEST_DATASET = 'LoneWolf'  # Using LoneWolf dataset for validation

# Input paths
RAW_MFT_PATH = BASE_DIR / f'data/validation/mft/{TEST_DATASET}-MFT'
USNJRNL_CSV_PATH = BASE_DIR / f'data/validation/usnjrnl/{TEST_DATASET}-UsnJrnl.csv'

# Output paths
MFT_OUTPUT_PATH = BASE_DIR / f'data/validation/processed/phase 0.5/{TEST_DATASET}_MFT.csv'
ENRICHED_USNJRNL_PATH = BASE_DIR / f'data/validation/processed/phase 0.5/{TEST_DATASET}_UsnJrnl_enriched.csv'

# Create output directory
os.makedirs(MFT_OUTPUT_PATH.parent, exist_ok=True)

print(f"Raw $MFT: {RAW_MFT_PATH}")
print(f"Existing UsnJrnl CSV: {USNJRNL_CSV_PATH}")
print(f"Output MFT CSV: {MFT_OUTPUT_PATH}")
print(f"Output Enriched UsnJrnl: {ENRICHED_USNJRNL_PATH}")


Raw $MFT: /Users/soni/Github/Digital-Detectives_Thesis/data/validation/mft/LoneWolf-MFT
Existing UsnJrnl CSV: /Users/soni/Github/Digital-Detectives_Thesis/data/validation/usnjrnl/LoneWolf-UsnJrnl.csv
Output MFT CSV: /Users/soni/Github/Digital-Detectives_Thesis/data/validation/processed/phase 0.5/LoneWolf_MFT.csv
Output Enriched UsnJrnl: /Users/soni/Github/Digital-Detectives_Thesis/data/validation/processed/phase 0.5/LoneWolf_UsnJrnl_enriched.csv


# Cell 2: Parse $MFT Using analyzeMFT
analyzeMFT is a Python tool that parses raw $MFT files and extracts file metadata including MACE timestamps, FileReferenceNumber, and file paths.

In [31]:
# Cell 2: Parse $MFT using analyzeMFT (async version)
import analyzeMFT
import sys
import asyncio

print("Parsing $MFT...")

async def parse_mft():
    # Set up command-line arguments for analyzeMFT
    original_argv = sys.argv.copy()
    
    try:
        sys.argv = [
            'analyzeMFT',
            '-f', str(RAW_MFT_PATH),
            '-o', str(MFT_OUTPUT_PATH),
            '--csv'
        ]
        
        # Await the async CLI main function
        await analyzeMFT.cli_main()
        
        print(f"Successfully parsed $MFT to {MFT_OUTPUT_PATH}")
        
    except Exception as e:
        print(f"Error parsing $MFT: {e}")
        import traceback
        traceback.print_exc()
    finally:
        # Restore original argv
        sys.argv = original_argv

# Run the async function
await parse_mft()

# Verify output file was created
import os
if os.path.exists(MFT_OUTPUT_PATH):
    file_size = os.path.getsize(MFT_OUTPUT_PATH)
    print(f"MFT CSV created: {file_size:,} bytes")
else:
    print(f"Warning: Output file not created at {MFT_OUTPUT_PATH}")


2025-12-29 23:49:01 - analyzeMFT.validators - WARNING - Output file already exists and will be overwritten: /Users/soni/Github/Digital-Detectives_Thesis/data/validation/processed/phase 0.5/LoneWolf_MFT.csv


Parsing $MFT...


2025-12-29 23:49:03 - analyzeMFT.validators - WARNING - Large attribute detected: type=144, length=688, record_size=1024
2025-12-29 23:49:03 - analyzeMFT.validators - WARNING - Large attribute detected: type=144, length=536, record_size=1024
2025-12-29 23:49:03 - analyzeMFT.validators - WARNING - Large attribute detected: type=144, length=632, record_size=1024
2025-12-29 23:49:03 - analyzeMFT.validators - WARNING - Large attribute detected: type=144, length=656, record_size=1024
2025-12-29 23:49:03 - analyzeMFT.validators - WARNING - Large attribute detected: type=144, length=552, record_size=1024
2025-12-29 23:49:03 - analyzeMFT.validators - WARNING - Large attribute detected: type=144, length=536, record_size=1024
2025-12-29 23:49:03 - analyzeMFT.validators - WARNING - Large attribute detected: type=128, length=536, record_size=1024
2025-12-29 23:49:03 - analyzeMFT.analyzer - ERROR - Attribute validation failed at record 230: Attribute length 24838176 exceeds record size 1024
2025-12

Successfully parsed $MFT to /Users/soni/Github/Digital-Detectives_Thesis/data/validation/processed/phase 0.5/LoneWolf_MFT.csv
MFT CSV created: 100,308,968 bytes


In [32]:
# Cell 3: Load and Inspect $MFT Data

# Load parsed $MFT
mft_df = pd.read_csv(MFT_OUTPUT_PATH, low_memory=False)

print(f"$MFT Records: {len(mft_df):,}")
print(f"\nColumns: {list(mft_df.columns)}")
print(f"\n$MFT Sample:")
mft_df.head()

$MFT Records: 147,500

Columns: ['Record Number', 'Record Status', 'Record Type', 'File Type', 'Sequence Number', 'Parent Record Number', 'Parent Record Sequence Number', 'Filename', 'Filepath', 'SI Creation Time', 'SI Modification Time', 'SI Access Time', 'SI Entry Time', 'FN Creation Time', 'FN Modification Time', 'FN Access Time', 'FN Entry Time', 'Object ID', 'Birth Volume ID', 'Birth Object ID', 'Birth Domain ID', 'Has Standard Information', 'Has Attribute List', 'Has File Name', 'Has Volume Name', 'Has Volume Information', 'Has Data', 'Has Index Root', 'Has Index Allocation', 'Has Bitmap', 'Has Reparse Point', 'Has EA Information', 'Has EA', 'Has Logged Utility Stream', 'Attribute List Details', 'Security Descriptor', 'Volume Name', 'Volume Information', 'Data Attribute', 'Index Root', 'Index Allocation', 'Bitmap', 'Reparse Point', 'EA Information', 'EA', 'Logged Utility Stream', 'MD5', 'SHA256', 'SHA512', 'CRC32']

$MFT Sample:


,Record Number,Record Status,Record Type,File Type,Sequence Number,Parent Record Number,Parent Record Sequence Number,Filename,Filepath,SI Creation Time,...,Index Allocation,Bitmap,Reparse Point,EA Information,EA,Logged Utility Stream,MD5,SHA256,SHA512,CRC32
0,0,Invalid,Not in Use,File,0,0,0,NaN,NaN,Not defined,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Valid,In Use,File,1,5,0,$MFTMirr,NaN,2018-03-27T13:07:33.477Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,Valid,In Use,File,2,5,0,$LogFile,NaN,2018-03-27T13:07:33.477Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,Valid,In Use,File,3,5,0,$Volume,NaN,2018-03-27T13:07:33.477Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,Valid,In Use,File,4,5,0,$AttrDef,NaN,2018-03-27T13:07:33.477Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Cell 4: Extract Relevant $MFT Columns
We need FileReferenceNumber (for joining) and MACE timestamps (for discrepancy detection).

In [33]:
# Cell 4: Extract Relevant $MFT Columns

# Actual columns from analyzeMFT v3.1.1:
# - Record Number: MFT record number
# - Filename: Filename
# - Filepath: Full file path  
# - SI Creation Time: $SI CreationTime
# - SI Modification Time: $SI ModifiedTime
# - SI Access Time: $SI AccessedTime
# - SI Entry Time: $SI MFTModifiedTime

# Select relevant columns and rename for clarity
mft_clean = mft_df[[
    'Record Number',
    'Filename',
    'Filepath',
    'SI Creation Time',
    'SI Modification Time',
    'SI Access Time',
    'SI Entry Time'
]].copy()

# Rename columns to match Oh et al.'s terminology
mft_clean.columns = [
    'FileReferenceNumber',
    'FileName_MFT',
    'FullPath_MFT',
    'SI_CreationTime',
    'SI_ModifiedTime',
    'SI_AccessedTime',
    'SI_MFTModifiedTime'
]

print(f"Cleaned $MFT records: {len(mft_clean):,}")
print(f"\nSample records:")
mft_clean.head(10)


Cleaned $MFT records: 147,500

Sample records:


,FileReferenceNumber,FileName_MFT,FullPath_MFT,SI_CreationTime,SI_ModifiedTime,SI_AccessedTime,SI_MFTModifiedTime
0,0,NaN,NaN,Not defined,Not defined,Not defined,Not defined
1,1,$MFTMirr,NaN,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z
2,2,$LogFile,NaN,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z
3,3,$Volume,NaN,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z
4,4,$AttrDef,NaN,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z
5,5,.,NaN,2017-09-29T08:45:11.680Z,2018-04-04T05:59:44.969Z,2018-04-04T05:59:44.969Z,2018-04-04T05:59:44.969Z
6,6,$Bitmap,NaN,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z
7,7,$Boot,NaN,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z
8,8,$BadClus,NaN,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z
9,9,$Secure,NaN,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z,2018-03-27T13:07:33.477Z


In [34]:
# Cell 5: Load UsnJrnl Data
# Load existing UsnJrnl CSV from Oh et al.'s tool
usnjrnl_df = pd.read_csv(USNJRNL_CSV_PATH)

print(f"UsnJrnl Records: {len(usnjrnl_df):,}")
print(f"\nColumns: {list(usnjrnl_df.columns)}")
print(f"\nUsnJrnl Sample:")
usnjrnl_df.head()


UsnJrnl Records: 352,849

Columns: ['TimeStamp(UTC+8)', 'USN', 'File/Directory Name', 'FullPath', 'EventInfo', 'SourceInfo', 'FileAttribute', 'Carving Flag', 'FileReferenceNumber', 'ParentFileReferenceNumber']

UsnJrnl Sample:


,TimeStamp(UTC+8),USN,File/Directory Name,FullPath,EventInfo,SourceInfo,FileAttribute,Carving Flag,FileReferenceNumber,ParentFileReferenceNumber
0,04/01/18 16:56:07:567,209715200,f498ac39e16a30c8_1,\Users\jcloudy\AppData\Local\Google\Chrome\Use...,Data_Added / Data_Overwritten / Data_Truncated,Normal,Archive,NaN,0x0007000000022A4A,0x000200000000572F
1,04/01/18 16:56:07:567,209715296,f498ac39e16a30c8_1,\Users\jcloudy\AppData\Local\Google\Chrome\Use...,Data_Added / Data_Overwritten / Data_Truncated...,Normal,Archive,NaN,0x0007000000022A4A,0x000200000000572F
2,04/01/18 16:56:08:568,209715392,000003.log,\Users\jcloudy\AppData\Local\Google\Chrome\Use...,Data_Added,Normal,Archive,NaN,0x0002000000005737,0x000200000000572E
3,04/01/18 16:56:08:568,209715472,3a9f2377dc054e52_0,\Users\jcloudy\AppData\Local\Google\Chrome\Use...,File_Created,Normal,Archive,NaN,0x00A900000001AF8C,0x000200000000572F
4,04/01/18 16:56:08:568,209715568,3a9f2377dc054e52_0,\Users\jcloudy\AppData\Local\Google\Chrome\Use...,File_Created / Data_Added,Normal,Archive,NaN,0x00A900000001AF8C,0x000200000000572F


# Cell 6: Prepare FileReferenceNumber for Joining
FileReferenceNumber in UsnJrnl is in hex format (e.g., '0x000B000000022A50'), while in $MFT it's decimal. We need to extract the record number portion.

In [35]:
# UsnJrnl FileReferenceNumber format: 0xSSSSSSSSRRRRRRRR
# Where SSSS = sequence number, RRRRRRRR = record number (lower 48 bits)

def parse_file_ref_number(hex_ref):
    """
    Extract MFT record number from FileReferenceNumber hex string.
    Format: 0xSSSSSSSSRRRRRRRR
    Returns: Record number (decimal)
    """
    if pd.isna(hex_ref) or hex_ref == '':
        return None
    
    # Remove '0x' prefix and convert to int
    hex_value = int(hex_ref, 16)
    
    # Extract lower 48 bits (record number)
    # This matches the MFT Record Number
    record_num = hex_value & 0xFFFFFFFFFFFF
    
    return record_num

# Apply to UsnJrnl
usnjrnl_df['MFT_RecordNumber'] = usnjrnl_df['FileReferenceNumber'].apply(parse_file_ref_number)

print(f"Parsed FileReferenceNumbers: {usnjrnl_df['MFT_RecordNumber'].notna().sum():,}")
print("\nSample mappings:")
print(usnjrnl_df[['FileReferenceNumber', 'MFT_RecordNumber']].head(10))


Parsed FileReferenceNumbers: 352,849

Sample mappings:
  FileReferenceNumber  MFT_RecordNumber
0  0x0007000000022A4A            141898
1  0x0007000000022A4A            141898
2  0x0002000000005737             22327
3  0x00A900000001AF8C            110476
4  0x00A900000001AF8C            110476
5  0x00A900000001AF8C            110476
6  0x00A900000001AF8C            110476
7  0x000F000000022A95            141973
8  0x000F000000022A95            141973
9  0x000F000000022A95            141973


In [36]:
# Cell 7: Join UsnJrnl with $MFT

# Left join: Keep all UsnJrnl records, add $MFT data where available
usnjrnl_enriched = usnjrnl_df.merge(
    mft_clean,
    left_on='MFT_RecordNumber',
    right_on='FileReferenceNumber',
    how='left',
    suffixes=('_event', '_mft')
)

# Check join success rate
total_records = len(usnjrnl_enriched)
matched_records = usnjrnl_enriched['SI_CreationTime'].notna().sum()
match_rate = (matched_records / total_records) * 100

print(f"Total UsnJrnl records: {total_records:,}")
print(f"Matched with $MFT: {matched_records:,} ({match_rate:.1f}%)")
print(f"Unmatched: {total_records - matched_records:,}")

usnjrnl_enriched.head()


# Filter to only rows where filenames match
name_match = (usnjrnl_enriched['File/Directory Name'] == usnjrnl_enriched['FileName_MFT'])
usnjrnl_enriched_matched = usnjrnl_enriched[name_match].copy()

print(f"Original records: {len(usnjrnl_enriched):,}")
print(f"Name-matched records: {len(usnjrnl_enriched_matched):,}")
print(f"Dropped (mismatched): {len(usnjrnl_enriched) - len(usnjrnl_enriched_matched):,}")


Total UsnJrnl records: 352,849
Matched with $MFT: 352,849 (100.0%)
Unmatched: 0
Original records: 352,849
Name-matched records: 101,023
Dropped (mismatched): 251,826


# Cell 8: Format Timestamps for Compatibility

Parse MFT timestamps and convert to same format as UsnJrnl (no milliseconds)
This makes them directly comparable in later phases

In [37]:
# Cell 8: Format Timestamps - All in UTC+8

# Parse MFT timestamps (they are in UTC, need to convert to UTC+8)
for col in ['SI_CreationTime', 'SI_ModifiedTime', 'SI_AccessedTime', 'SI_MFTModifiedTime']:
    if col in usnjrnl_enriched.columns:
        # Parse as UTC
        usnjrnl_enriched[f'{col}_UTC'] = pd.to_datetime(
            usnjrnl_enriched[col],
            utc=True,
            errors='coerce'
        ).dt.tz_localize(None)
        
        # Convert to UTC+8 (add 8 hours)
        usnjrnl_enriched[col] = usnjrnl_enriched[f'{col}_UTC'] + pd.Timedelta(hours=8)
        
        # Format as MM/DD/YYYY HH:MM:SS.mmm
        usnjrnl_enriched[f'{col}_Formatted'] = usnjrnl_enriched[col].dt.strftime('%m/%d/%Y %H:%M:%S.%f').str[:-3]

# Parse UsnJrnl timestamp (already in UTC+8, just parse)
# Format: "04/01/18 16:56:07:567" = MM/DD/YY HH:MM:SS:mmm
usnjrnl_enriched['EventTime'] = pd.to_datetime(
    usnjrnl_enriched['TimeStamp(UTC+8)'],
    format='%m/%d/%y %H:%M:%S:%f',
    errors='coerce'
)

# Format EventTime
usnjrnl_enriched['EventTime_Formatted'] = usnjrnl_enriched['EventTime'].dt.strftime('%m/%d/%Y %H:%M:%S.%f').str[:-3]

print("All timestamps now in UTC+8 with format: MM/DD/YYYY HH:MM:SS.mmm")
print(f"\nVerification with HoldMyTidePod.jpg:")
sample = usnjrnl_enriched[usnjrnl_enriched['File/Directory Name'] == 'HoldMyTidePod.jpg']
if len(sample) > 0:
    print("\nUsnJrnl Event Time (UTC+8):", sample['EventTime_Formatted'].iloc[0])
    print("MFT Creation Time (UTC+8): ", sample['SI_CreationTime_Formatted'].iloc[0])
    print("\nShould match Oh et al. detection:")
    print("  CreationTime:        2018-03-30 11:29:20")
    print("  Creation Event Time: 2018-04-05 10:21:03")


All timestamps now in UTC+8 with format: MM/DD/YYYY HH:MM:SS.mmm

Verification with HoldMyTidePod.jpg:

UsnJrnl Event Time (UTC+8): 04/02/2018 09:12:36.123
MFT Creation Time (UTC+8):  03/30/2018 11:29:20.579

Should match Oh et al. detection:
  CreationTime:        2018-03-30 11:29:20
  Creation Event Time: 2018-04-05 10:21:03


In [38]:
# Cell 8.5: Standardize All Timestamp Formats to MM/DD/YYYY HH:MM:SS.mmm

# MFT timestamps already formatted in Cell 8, just verify EventTime
if 'EventTime' in usnjrnl_enriched.columns:
    usnjrnl_enriched['EventTime_Formatted'] = usnjrnl_enriched['EventTime'].dt.strftime('%m/%d/%Y %H:%M:%S.%f').str[:-3]

print("Standardized timestamp formats (all in UTC+8):")
print("\nColumns created:")
for col in usnjrnl_enriched.columns:
    if '_Formatted' in col:
        print(f"  {col}")

# Verify with HoldMyTidePod.jpg
sample = usnjrnl_enriched[usnjrnl_enriched['File/Directory Name'] == 'HoldMyTidePod.jpg']
if len(sample) > 0:
    print(f"\nVerification - HoldMyTidePod.jpg:")
    print(f"  Original TimeStamp(UTC+8):        {sample['TimeStamp(UTC+8)'].iloc[0]}")
    print(f"  EventTime_Formatted (UTC+8):      {sample['EventTime_Formatted'].iloc[0]}")
    print(f"  SI_CreationTime_Formatted (UTC+8): {sample['SI_CreationTime_Formatted'].iloc[0]}")
    print(f"\n  Expected from Oh et al. detection:")
    print(f"    Creation Event Time: 2018-04-05 10:21:03")
    print(f"    CreationTime:        2018-03-30 11:29:20")
else:
    print("\nHoldMyTidePod.jpg not found in dataset - showing first row:")
    if len(usnjrnl_enriched) > 0:
        first_row = usnjrnl_enriched.iloc[0]
        print(f"  File: {first_row['File/Directory Name']}")
        print(f"  TimeStamp(UTC+8):     {first_row['TimeStamp(UTC+8)']}")
        if 'EventTime_Formatted' in first_row:
            print(f"  EventTime_Formatted:  {first_row['EventTime_Formatted']}")
        if 'SI_CreationTime_Formatted' in first_row:
            print(f"  SI_CreationTime_Formatted: {first_row['SI_CreationTime_Formatted']}")


Standardized timestamp formats (all in UTC+8):

Columns created:
  SI_CreationTime_Formatted
  SI_ModifiedTime_Formatted
  SI_AccessedTime_Formatted
  SI_MFTModifiedTime_Formatted
  EventTime_Formatted

Verification - HoldMyTidePod.jpg:
  Original TimeStamp(UTC+8):        04/02/18 09:12:36:1236
  EventTime_Formatted (UTC+8):      04/02/2018 09:12:36.123
  SI_CreationTime_Formatted (UTC+8): 03/30/2018 11:29:20.579

  Expected from Oh et al. detection:
    Creation Event Time: 2018-04-05 10:21:03
    CreationTime:        2018-03-30 11:29:20


# Cell 9: Sanity Check

In [39]:
# Cell 9: Quick Sanity Check - All in UTC+8

creation_events = usnjrnl_enriched[
    usnjrnl_enriched['EventInfo'].str.contains('File_Created', na=False)
].copy()

# Calculate time difference (both now in UTC+8)
creation_events['Time_Diff_Seconds'] = abs(
    (creation_events['SI_CreationTime'] - creation_events['EventTime']).dt.total_seconds()
)

print(f"File_Created events: {len(creation_events):,}")
print(f"\nTime difference statistics (comparing UTC+8 to UTC+8):")
print(creation_events['Time_Diff_Seconds'].describe())

# Show a few examples of matches and mismatches
print(f"\nExamples of close matches (diff < 60s):")
close_matches = creation_events[creation_events['Time_Diff_Seconds'] < 60].head(3)
print(close_matches[['File/Directory Name', 'EventTime', 'SI_CreationTime', 'Time_Diff_Seconds']].to_string())

print(f"\nExamples of large differences (diff > 1 day):")
large_diff = creation_events[creation_events['Time_Diff_Seconds'] > 86400].head(3)
print(large_diff[['File/Directory Name', 'EventTime', 'SI_CreationTime', 'Time_Diff_Seconds']].to_string())

# Verify HoldMyTidePod.jpg matches Oh et al.
print(f"\n" + "="*60)
print("Verification - HoldMyTidePod.jpg:")
print("="*60)
holdmytidepod = creation_events[creation_events['File/Directory Name'] == 'HoldMyTidePod.jpg']
if len(holdmytidepod) > 0:
    print(f"Event Time (UTC+8):    {holdmytidepod['EventTime_Formatted'].iloc[0]}")
    print(f"Creation Time (UTC+8): {holdmytidepod['SI_CreationTime_Formatted'].iloc[0]}")
    print(f"Time difference:       {holdmytidepod['Time_Diff_Seconds'].iloc[0]:,.0f} seconds ({holdmytidepod['Time_Diff_Seconds'].iloc[0]/86400:.1f} days)")
    print(f"\nExpected from Oh et al. detection:")
    print(f"  Creation Event Time: 2018-04-05 10:21:03")
    print(f"  CreationTime:        2018-03-30 11:29:20")
    print(f"  Difference: ~5.9 days")
else:
    print("HoldMyTidePod.jpg not found in File_Created events")

print(f"\nNote: Full timestamp discrepancy detection will be done in Phase 2 Feature Engineering")


File_Created events: 146,838

Time difference statistics (comparing UTC+8 to UTC+8):
count    1.453680e+05
mean     1.028598e+06
std      3.847519e+06
min      0.000000e+00
25%      4.292930e+01
50%      4.474548e+03
75%      1.101851e+05
max      1.046616e+08
Name: Time_Diff_Seconds, dtype: float64

Examples of close matches (diff < 60s):
  File/Directory Name               EventTime         SI_CreationTime  Time_Diff_Seconds
3  3a9f2377dc054e52_0 2018-04-01 16:56:08.568 2018-04-01 16:56:08.772              0.204
4  3a9f2377dc054e52_0 2018-04-01 16:56:08.568 2018-04-01 16:56:08.772              0.204
5  3a9f2377dc054e52_0 2018-04-01 16:56:08.568 2018-04-01 16:56:08.772              0.204

Examples of large differences (diff > 1 day):
   File/Directory Name               EventTime         SI_CreationTime  Time_Diff_Seconds
18  79ba41129dbeffed_0 2018-04-01 16:56:08.568 2018-04-04 12:33:40.276         243451.708
19  79ba41129dbeffed_0 2018-04-01 16:56:08.568 2018-04-04 12:33:40.276     

In [40]:
# Cell 10: Save Enriched UsnJrnl with Proper Formatting

# CRITICAL FIX: Rename FileReferenceNumber_event back to FileReferenceNumber
# This was renamed during merge due to suffixes parameter
if 'FileReferenceNumber_event' in usnjrnl_enriched.columns:
    usnjrnl_enriched = usnjrnl_enriched.rename(columns={'FileReferenceNumber_event': 'FileReferenceNumber'})
    print("Renamed FileReferenceNumber_event → FileReferenceNumber")

# Select output columns - keep ALL UsnJrnl columns + MFT additions
output_columns = [
    # Original UsnJrnl columns (DON'T DROP ANY)
    'TimeStamp(UTC+8)', 'USN', 'File/Directory Name', 'FullPath', 'EventInfo',
    'SourceInfo', 'FileAttribute', 'Carving Flag', 'FileReferenceNumber', 'ParentFileReference',  # KEEP THIS

    # Added from $MFT
    'FileName_MFT', 
    'SI_CreationTime_Formatted', 'SI_ModifiedTime_Formatted', 
    'SI_AccessedTime_Formatted', 'SI_MFTModifiedTime_Formatted',
    'EventTime_Formatted',
    'MFT_RecordNumber'
]

# Keep only columns that exist
output_columns = [col for col in output_columns if col in usnjrnl_enriched.columns]
usnjrnl_output = usnjrnl_enriched[output_columns].copy()

# Drop FullPath_MFT if it's empty
if 'FullPath_MFT' in usnjrnl_output.columns and usnjrnl_output['FullPath_MFT'].isna().all():
    usnjrnl_output = usnjrnl_output.drop(columns=['FullPath_MFT'])
    print("Dropped FullPath_MFT (empty)")

# Save to CSV
usnjrnl_output.to_csv(ENRICHED_USNJRNL_PATH, index=False)

print(f"Saved: {ENRICHED_USNJRNL_PATH}")
print(f"Records: {len(usnjrnl_output):,}")
print(f"Columns: {list(usnjrnl_output.columns)}")
print(f"\nVerify FileReferenceNumber is present: {'FileReferenceNumber' in usnjrnl_output.columns}")
print(f"\nTo view properly in Excel:")
print("  1. Open Excel → Data → From Text/CSV")
print("  2. Select all timestamp columns → Change Type → Text")
print("  Or use VS Code to view without corruption")

Renamed FileReferenceNumber_event → FileReferenceNumber
Saved: /Users/soni/Github/Digital-Detectives_Thesis/data/validation/processed/phase 0.5/LoneWolf_UsnJrnl_enriched.csv
Records: 352,849
Columns: ['TimeStamp(UTC+8)', 'USN', 'File/Directory Name', 'FullPath', 'EventInfo', 'SourceInfo', 'FileAttribute', 'Carving Flag', 'FileReferenceNumber', 'FileName_MFT', 'SI_CreationTime_Formatted', 'SI_ModifiedTime_Formatted', 'SI_AccessedTime_Formatted', 'SI_MFTModifiedTime_Formatted', 'EventTime_Formatted', 'MFT_RecordNumber']

Verify FileReferenceNumber is present: True

To view properly in Excel:
  1. Open Excel → Data → From Text/CSV
  2. Select all timestamp columns → Change Type → Text
  Or use VS Code to view without corruption


In [41]:
import pandas as pd

# Load the CSV
path = "/Users/soni/Github/Digital-Detectives_Thesis/data/validation/processed/phase 0.5/LoneWolf_UsnJrnl_enriched.csv"
df = pd.read_csv(path)

# Show all rows when printing
pd.set_option('display.max_rows', None)

# Count instances of each EventInfo value
event_counts = df["EventInfo"].value_counts()
print(event_counts)


EventInfo
File_Created                                                                                                                      54905
File_Closed / File_Deleted                                                                                                        48184
File_Created / Data_Added                                                                                                         29402
File_Created / Data_Added / File_Closed                                                                                           24698
File_Created / File_Closed                                                                                                        22690
Data_Added                                                                                                                        14937
Data_Added / File_Closed                                                                                                          11783
Basic_Info_Changed                    